<a href="https://colab.research.google.com/github/MayaVolkov/astronaut-missions-analysis/blob/main/Astronaut_Mission_Analysis_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Astronaut Mission Analysis (from 1961 to 2026)

## Question
How has the number of countries sending people to space changed since 1961?

Sub-questions:
1. How do countries without their own human launch rockets get their astronauts there?
2. How has participation in human spaceflight by countries outside the three launch nations changed since 1961?

## Data:
My data is coming from two sources: Mission data from 1961 to early 2020 is available through a public database published by Mariya Stavnichuk and Tatsuya Corlett in 2020. I got this dataset from TidyTuesday, a public github repo that publishes clean datasets for people to practice on. This dataset only goes until 2020, and I to answer my question I need the most curennt data as well. For this reason I am using a second dataset, for the Missions flown between 2021 and 2026 I will be using the public Wikipedia page to get information on flights happening after early 2020 into 2021.

## Scope:
I have chosen to include all orbital flights between 1961 and 2026. 1961 is when the first human space mission occured, and going until 2026 will allow me to get the most up to date data. I chose not to include suborbital flights as based on my research those are typically commerical spaceflights that arent tied to a nation or space research institute and can be attributed to "space tourism".

## Why does this matter?
This matters because space is one of the last frontiers of truly wilderness and unknown, and making space technology more accessible to other nations allows for there to be more minds and global energy directed into research into space.

In [40]:
import pandas as pd #tables (DataFrames)

import requests #download the Wikipedia page

from io import StringIO #lets read_html treat the HTML string like a file

import re #regex, to rescue countries and separators from the HTML

Below I am storing the link to the data for the missions between 1961 and early 2020. I'm storing it in a variable so that it's easy for me to grab whenever I need. Instead of copying in the whole link over again, I can just type the variable.

In [41]:
data_url_until_2020 = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-07-14/astronauts.csv"

Using Pandas to load in the csv file and turn it into a table that colab can read. Then I print the first five rows of the table as a sanity check.

In [42]:
data_table_2020 = pd.read_csv(data_url_until_2020)
#previewing the first five rows
data_table_2020.head()

,id,number,nationwide_number,name,original_name,sex,year_of_birth,nationality,military_civilian,selection,...,year_of_mission,mission_title,ascend_shuttle,in_orbit,descend_shuttle,hours_mission,total_hrs_sum,field21,eva_hrs_mission,total_eva_hrs
0,1,1,1,"Gagarin, Yuri",ГАГАРИН Юрий Алексеевич,male,1934,U.S.S.R/Russia,military,TsPK-1,...,1961,Vostok 1,Vostok 1,Vostok 2,Vostok 3,1.77,1.77,0,0.0,0.0
1,2,2,2,"Titov, Gherman",ТИТОВ Герман Степанович,male,1935,U.S.S.R/Russia,military,TsPK-1,...,1961,Vostok 2,Vostok 2,Vostok 2,Vostok 2,25.00,25.30,0,0.0,0.0
2,3,3,1,"Glenn, John H., Jr.","Glenn, John H., Jr.",male,1921,U.S.,military,NASA Astronaut Group 1,...,1962,MA-6,MA-6,MA-6,MA-6,5.00,218.00,0,0.0,0.0
3,4,3,1,"Glenn, John H., Jr.","Glenn, John H., Jr.",male,1921,U.S.,military,NASA Astronaut Group 2,...,1998,STS-95,STS-95,STS-95,STS-95,213.00,218.00,0,0.0,0.0
4,5,4,2,"Carpenter, M. Scott","Carpenter, M. Scott",male,1925,U.S.,military,NASA- 1,...,1962,Mercury-Atlas 7,Mercury-Atlas 7,Mercury-Atlas 7,Mercury-Atlas 7,5.00,5.00,0,0.0,0.0


I'm going to be focusing on analyzing the following four columns of data for the following reasons:
- nationality: in order to find the countries who are sending people to space
- name: (1) to count unique people instead of rows, since repeat flyers have multiple rows, and (2) to match astronauts when you combine the two datasets.
- year_of_mission: im checking the changes of countries over time
- ascend_shuttle: tells me which spacecraft launched that person so I know which country's rocket carried them.  


research decisions:
- Only count astronauts who reached orbit (aborted launches and suborbital flights excluded), because the question is about arrivals, not attempts.
- Treat the USSR and Russia as one country, since they never overlapped and share one continuous space program.
- Soviet Astronauts labelled as U.S.S.R/Ukraine will count for Ukraine, while soldiers labelled as U.S.S.R/Russia will count for Russia (for justification, see below)

In [43]:
analysis_columns = ["name", "year_of_mission", "nationality", "ascend_shuttle"] # the four columns my research question needs, previewed

data_table_2020[analysis_columns].head(5)


,name,year_of_mission,nationality,ascend_shuttle
0,"Gagarin, Yuri",1961,U.S.S.R/Russia,Vostok 1
1,"Titov, Gherman",1961,U.S.S.R/Russia,Vostok 2
2,"Glenn, John H., Jr.",1962,U.S.,MA-6
3,"Glenn, John H., Jr.",1998,U.S.,STS-95
4,"Carpenter, M. Scott",1962,U.S.,Mercury-Atlas 7


####Research decision: how to count Soviet-era nationality labels.
 The U.S.S.R/Ukraine row (Kadenyuk, 1997) is counted as Ukraine. His flight came after Ukraine’s independence in 1991, he flew as a citizen of Ukraine, and he flew on a U.S. spacecraft, so it isn’t a continuation of the Soviet program.

In [44]:
# check: Kadenyuk is the only row labeled U.S.S.R/Ukraine
ukraine = data_table_2020[data_table_2020["nationality"] == "U.S.S.R/Ukraine"]
ukraine[["name", "year_of_mission", "ascend_shuttle"]]

,name,year_of_mission,ascend_shuttle
913,"Kadenyuk, Leonid",1997,STS87


## Checking for missing data

Before going further, I'm checking whether any of my four research columns have blank cells, since a missing nationality or year would silently drop an astronaut from my counts.

Seeing what cells are empty in the whole table:

In [45]:
data_table_2020.isnull().sum()

,0
id,0
number,0
nationwide_number,0
name,0
original_name,5
sex,0
year_of_birth,0
nationality,0
military_civilian,0
selection,5


seeing what cells are missing just from the columns im using for my research

In [46]:
data_table_2020[analysis_columns].isnull().sum()

,0
name,0
year_of_mission,0
nationality,0
ascend_shuttle,1


The only column with missing data is ascend_shuttle, which has one blank cell in its column. Because there is only one blank cell I will keep that row and manually fill it in.

In [47]:
# show the one row with a blank ascend_shuttle
data_table_2020[data_table_2020["ascend_shuttle"].isnull()][analysis_columns]

,name,year_of_mission,nationality,ascend_shuttle
197,Bertalan Farkas,1980,Hungry,NaN


The blank ascend_shuttle cell belongs to Bertalan Farkas, Hungary’s first astronaut, who flew in 1980. Since it’s the only blank in my research columns, I’m filling it in by hand rather than dropping the row. Farkas launched on Soyuz 36 and returned to Earth on Soyuz 35, so his ascend_shuttle value is Soyuz 36. Source: https://www.britannica.com/biography/Bertalan-Farkas

In [48]:
# fill in Farkas's launch spacecraft (source: Britannica, linked above)
data_table_2020.loc[data_table_2020["name"] == "Bertalan Farkas", "ascend_shuttle"] = "Soyuz 36"

# check: should now print 0
data_table_2020["ascend_shuttle"].isnull().sum()

np.int64(0)

In [49]:
no_comma_count = (~(data_table_2020["name"].str.contains(","))).sum()
no_comma_count

np.int64(33)

In [50]:
# the names with no comma, so I can see what patterns they follow
without_comma = data_table_2020[~data_table_2020["name"].str.contains(",")]
without_comma[["name", "nationality"]]

,name,nationality
67,Jugderdemidiin Gurragchaa,Mongolia
72,Dumitru Prunariu,Romania
185,Vladimir Remek,Czechoslovakia
188,Miroslaw Hermaszewske,Poland
193,Georgi Ivanov,Bulgaria
197,Bertalan Farkas,Hungry
200,Pham Tuan,Vietnam
201,Arnaldo Tamayo Mendez,Cuba
285,Rakesh Sharma,India
570,Helen Sharman,U.K.


In [51]:
# fix the three clear errors found above: name order, a typo, an alternate spelling
data_table_2020["name_clean"] = (
    data_table_2020["name"]
    .str.replace("Kregel Kevin R.", "Kevin R. Kregel")
    .str.replace("Hermaszewske", "Hermaszewski")
    .str.replace(" (Aydyn) ", " ")
)

###Decision: how to handle names
Keep names as the dataset has them. I only fixed clear errors (name order, typos, and an alternate spelling in parentheses). I did not add or remove middle names, middle initials, or patronymics, because changing them would mean creating versions of names that the source doesn’t use. Names will have different numbers of parts (for example, Yuri Vladimirovich Usachyov vs. Helen Sharman), but that’s fine: when I link datasets later, the match key will only use the first and last parts, so whatever’s in the middle won’t affect matching.

In [52]:
changed = data_table_2020["name"] != data_table_2020["name_clean"]
data_table_2020[changed][["name", "name_clean"]]

,name,name_clean
188,Miroslaw Hermaszewske,Miroslaw Hermaszewski
807,Kregel Kevin R.,Kevin R. Kregel
808,Kregel Kevin R.,Kevin R. Kregel
809,Kregel Kevin R.,Kevin R. Kregel
810,Kregel Kevin R.,Kevin R. Kregel
1253,Aidyn (Aydyn) Akanovich Aimbetov,Aidyn Akanovich Aimbetov


Flipped ‘Last, First’ names to ‘First Last’so the format matches Wikipedia’s ‘First Last’ names when I combine the two datasets. Names with a suffix (like Jr.) keep it at the end. Names without a comma were left as they were.

In [53]:
# which names are in "Last, First" format
has_comma = data_table_2020["name_clean"].str.contains(",")

# split each name at the comma: "Glenn, John H., Jr." → ["Glenn", "John H.", "Jr."]
pieces = data_table_2020["name_clean"].str.split(", ")

last = pieces.str[0]
first = pieces.str[1]
suffix = pieces.str[2].fillna("")   # most names have no suffix, so fill the blank with ""

# rebuild as "First Last Suffix"; strip() removes the extra space when there's no suffix
flipped = (first + " " + last + " " + suffix).str.strip()

# only use the flipped version where the name had a comma; otherwise keep it as is
data_table_2020["name_clean"] = flipped.where(has_comma, data_table_2020["name_clean"])

In [54]:
data_table_2020[["name", "name_clean"]].head(10) # check: before and after for the first 10 rows

,name,name_clean
0,"Gagarin, Yuri",Yuri Gagarin
1,"Titov, Gherman",Gherman Titov
2,"Glenn, John H., Jr.",John H. Glenn Jr.
3,"Glenn, John H., Jr.",John H. Glenn Jr.
4,"Carpenter, M. Scott",M. Scott Carpenter
5,"Nikolayev, Andriyan",Andriyan Nikolayev
6,"Nikolayev, Andriyan",Andriyan Nikolayev
7,"Popovich, Pavel",Pavel Popovich
8,"Popovich, Pavel",Pavel Popovich
9,"Schirra, Walter M., Jr.",Walter M. Schirra Jr.


In [55]:
data_table_2020["name_clean"].str.contains(",").sum() # check: 0 names still have a comma, so every "Last, First" name was flipped

np.int64(0)

## Getting the Wikipedia data (2021–2026):
Source: Wikipedia, “List of human spaceflights, 2021–present". Each mission is a table row (`<tr>`), with astronaut names inside links (`<a>`). Suborbital flights are marked with a green background (style="background:#BBFFBB") and will be excluded. Names include a flight count in parentheses, e.g. “Oleg Novitsky (3)”, which will need to be removed.

Creating a dictionary that stores the key User-Agent and my contact info so I can request information from Wikipedia because Wikipedia asks code to identify itself before it sends pages.

In [56]:
headers = {"User-Agent" : "Astronaut Mission Analysis Student Project (https://github.com/MayaVolkov)"}

Saving the Wiki URL and then requesting the information inside the page, attaching User-Agent to the request so Wikipedia knows where it's coming from. status_code reports if we got the file -> 200 means success.

In [57]:
wiki_url = "https://en.wikipedia.org/wiki/List_of_human_spaceflights,_2021%E2%80%93present"
response = requests.get(wiki_url, headers=headers)
response.status_code

200

In [58]:
# wrap the HTML text so read_html can treat it like a file
wrapped_html = StringIO(response.text)

# pull every table on the page into a list of DataFrames
tables = pd.read_html(wrapped_html)
print(len(tables))   # 7 tables on the page

7


In [59]:
# print each table's size to find which one is the mission list
for i, table in enumerate(tables):
    print(i, table.shape)

0 (76, 7)
1 (22, 2)
2 (8, 2)
3 (4, 2)
4 (1, 2)
5 (1, 2)
6 (1, 2)


Table 0 is the mission list (76 rows); the other six are small summary tables

In [60]:
missions = tables[0]
missions.columns

Index(['#', 'Crew', 'Launch spacecraft', 'Habitation', 'Habitation.1',
       'Return spacecraft', 'Brief mission summary'],
      dtype='object')

In [61]:
missions.head()

,#,Crew,Launch spacecraft,Habitation,Habitation.1,Return spacecraft,Brief mission summary
0,329,Oleg Novitsky (3),9 April 2021 Soyuz MS-18,ISS (crew 64/65),ISS (crew 64/65),17 October 2021 Soyuz MS-18,ISS crew rotation.
1,329,Pyotr Dubrov (1) Mark T. Vande Hei (2),9 April 2021 Soyuz MS-18,ISS (crew 64/65/66),ISS (crew 64/65/66),30 March 2022 Soyuz MS-19,ISS crew rotation.
2,330,Shane Kimbrough (3) K. Megan McArthur (2) Ak...,"23 April 2021 SpaceX Crew-2, Endeavour",ISS (crew 65/66),ISS (crew 65/66),"9 November 2021 SpaceX Crew-2, Endeavour",ISS crew rotation.
3,—,Frederick W. Sturckow Dave Mackay,22 May 2021 Unity 21,22 May 2021 Unity 21,22 May 2021 Unity 21,22 May 2021 Unity 21,First human spaceflight from the State of New ...
4,331,Nie Haisheng (3) Liu Boming (2) Tang Hongbo (1),17 June 2021 Shenzhou 12,TSS,TSS,17 September 2021 Shenzhou 12,First crew to Tiangong Space Station.


I used pd.read_html to turn the HTML into tables (DataFrames). Looking at the tables it returned, I knew the mission data was most likely in table[0], because it had the most columns (the most types of information) and the most rows (the most data).

pd.read_html throws away anything that isn't text, like the cells' background color or the flags next to crew members' names. I confirmed this hypothesis by running missions.head(), and found 3 surprises:

1. **Two Habitation headers:** The Habitation header on the page sits over two columns, so pandas needs two column names. It can't have two columns both called "Habitation," so it adds .1 to the second one to keep the names unique.
2. **Split rows sharing a #:** Some crew members launched together but returned on different spacecraft. Wikipedia gives each return group its own row and copies the launch info (and the #) into both, so one launch can appear as two rows.
3. **The "–" on some flights:** These, along with same-day launch and return dates, are a clue that these rows are suborbital flights. That's useful, since pd.read_html threw away the other signal (the green background).

Fix: before running read_html, I rewrite the HTML so the lost information becomes visible text. Each flag becomes [Country], and each line break between astronauts becomes a | separator.

In [62]:
# pattern for a flag: grabs the country name from the title attribute
flag_pattern = r'<[^>]*title="([^"]+)"[^>]*><img[^>]*>'
# pattern for a line break tag, with or without attributes
br_pattern = r'<br[^>]*>'

# replace each flag with "[Country] " and each line break with " | "
html_clean = re.sub(flag_pattern, r'[\1] ', response.text)
html_clean = re.sub(br_pattern, ' | ', html_clean)

In [63]:
# re-read the page from the cleaned HTML; this replaces the earlier missions table
tables_clean = pd.read_html(StringIO(html_clean))
missions = tables_clean[0]
missions.head()

,#,Crew,Launch | spacecraft,Habitation,Habitation.1,Return | spacecraft,Brief mission summary
0,329,[Russia] Oleg Novitsky (3),9 April 2021 | [Russia] Soyuz MS-18,ISS | (crew 64/65),ISS | (crew 64/65),17 October 2021 | [Russia] Soyuz MS-18,ISS crew rotation.
1,329,[Russia] Pyotr Dubrov (1) | [United States] Ma...,9 April 2021 | [Russia] Soyuz MS-18,ISS | (crew 64/65/66),ISS | (crew 64/65/66),30 March 2022 | [Russia] Soyuz MS-19,ISS crew rotation.
2,330,[United States] Shane Kimbrough (3) | [United ...,"23 April 2021 | [United States] SpaceX Crew-2,...",ISS | (crew 65/66),ISS | (crew 65/66),9 November 2021 | [United States] SpaceX Crew-...,ISS crew rotation.
3,—,[United States] Frederick W. Sturckow [United ...,22 May 2021 [United States] Unity 21,22 May 2021 [United States] Unity 21,22 May 2021 [United States] Unity 21,22 May 2021 [United States] Unity 21,First human spaceflight from the State of New ...
4,331,[China] Nie Haisheng (3) | [China] Liu Boming ...,17 June 2021 | [China] Shenzhou 12,TSS,TSS,17 September 2021 | [China] Shenzhou 12,First crew to Tiangong Space Station.


The fix worked: countries now appear as [Country] and astronauts are separated by |. Side effect: line breaks in other cells and headers also became pipes (e.g. the column is now ‘Launch | spacecraft’). I’ll account for that when I use those columns.

In [64]:
# remove flight counts like " (3)" from every name; keep the original Crew column untouched
missions["crew_clean"] = missions["Crew"].str.replace(r'\s*\(\d+\)', '', regex=True)
missions[["Crew", "crew_clean"]].head()

,Crew,crew_clean
0,[Russia] Oleg Novitsky (3),[Russia] Oleg Novitsky
1,[Russia] Pyotr Dubrov (1) | [United States] Ma...,[Russia] Pyotr Dubrov | [United States] Mark T...
2,[United States] Shane Kimbrough (3) | [United ...,[United States] Shane Kimbrough | [United Stat...
3,[United States] Frederick W. Sturckow [United ...,[United States] Frederick W. Sturckow [United ...
4,[China] Nie Haisheng (3) | [China] Liu Boming ...,[China] Nie Haisheng | [China] Liu Boming | [C...


In [65]:
# compare: how many rows each version of dropna would keep
print("now:", len(missions))
print("dropna():", len(missions.dropna()))
print('dropna(how="all"):', len(missions.dropna(how="all")))

now: 76
dropna(): 67
dropna(how="all"): 73


I chose dropna(how="all") because it only drops a row when every column is blank, meaning the row is completely empty. Plain dropna() would have been too severe: it would have dropped 9 rows instead of 3. The extra 6 rows were real missions with blank cells in columns like Habitation, which are blank because those missions never docked with a station. Habitation isn't relevant to my research, so it's fine for it to have empty cells. Had I used plain dropna(), rows that held the information I do need (names, nationality, year) would have been dropped just for missing Habitation, and I would have lost real astronauts from my data.

In [66]:
# 1. drop rows where every column is blank (spacer rows from the Wikipedia table)
missions = missions.dropna(how="all").copy()
# 2. trim stray pipes and spaces off the ends of each crew cell
missions["crew_clean"] = missions["crew_clean"].str.strip(" |")
# 3. remove Wikipedia footnote markers like [1] and [a]
footnotes = ["[1]", "[2]", "[3]", "[a]", "[b]"]
for note in footnotes:
    missions["crew_clean"] = missions["crew_clean"].str.replace(note, "", regex=False)
# 4. split each crew into a list of astronauts
missions["crew_list"] = missions["crew_clean"].str.split(" | ", regex=False)

The check goes through each row and counts the opening brackets ([) and pipes (|) in its crew cell. Every astronaut has a country tag, so brackets stand in for people, and pipes are the separators between them. A “healthy” cell should have one more bracket than pipes. Using the “not equal to” operator (!=), the check prints any row that doesn’t match this rule, meaning it’s “unhealthy” and needs a closer look.

In [67]:
# check: flag any crew cell where brackets ≠ pipes + 1
for i in missions.index:
    cell = missions.loc[i, "crew_clean"]
    if cell.count("[") != cell.count("|") + 1:
        print(i, cell)

3 [United States] Frederick W. Sturckow [United Kingdom] Dave Mackay
17 [United States] / [Spain] Michael López-Alegría | [United States] Larry Connor | [Canada] Mark Pathy | [Israel] Eytan Stibbe
35 [United States] Michael Masucci | [United States] Beth Moses | [United Kingdom] Adrian Reynard | [Italy] Nicola Pecile | [United States] Ken Baxter | [United Kingdom] / [South Africa] Timothy Nash
41 [United States] / [Spain] Michael López-Alegría | [Italy] Walter Villadei | [Turkey] Alper Gezeravcı | [Sweden] / [Norway] Marcus Wandt
42 [United States] Frederick Sturckow | [Italy] Nicola Pecile | [Ukraine] /[United States] Lina Borozdina | [United States] Robie Vaughn | [Austria] Franz Haider | [United States] Neil Kornswiet
49 [Italy] Nicola Pecile | [Canada] Jameel Janjua | [Turkey] Tuva Cihangir Atasever | [Italy] Giorgio Manenti | [United States] [Israel] Irving Pergament | [United States] Andy Sadhwani
50 [United Kingdom] /[Singapore] Nicolina Elrick | [United States] Karsen Kitchen |

Based on the check, I can group these 15 rows into 3 flagged groups:
1. One person, two country tags (with or without /). Includes rows 17, 35, 41, 42, 49, 50, 58, 60, and 70
2. Brackets that aren't countries ([1], [a], [b]). Includes rows 35 and 58
3. Two or more people with no | between them. Includes rows 3, 64, 66, 67, 70, and 71

Group 1 isn't wrong at all; the split is perfect. For example, in row 17, López-Alegría still ends up as one item in the list. The only thing that was wrong was the check: it assumed one bracket per person. This is called a "false alarm." This raises a research question to flag for later: if a person has two nationalities, do I count both or just one, and if just one, which one stays?

Group 2 is in between. The pipes are fine, so the split works. But [1], [a], [b] stay glued to names, like Adrian Reynard[1]. That will cause trouble later, when I pull the country out of the brackets (is "1" a country?).

Group 3 breaks the data. No | means .split() has nothing to cut on, so the whole crew stays as one item. Row 64 would count as one "astronaut" named Allie Kuehner[United States] Carl Kuehner[United States]..., and five real people would disappear from my count. An important find: all six rows in this group are suborbital (same launch and return date), so they will be excluded and don't need fixing. Important note: my "—" hypothesis was inaccurate, since only 1 of the 6 rows had a "—" in its # column. So I'll identify suborbital flights by same-day launch and return dates instead.

In [68]:
# check: are all the missing-pipe rows suborbital?
missions.loc[[3, 64, 66, 67, 70, 71], ["#", "Launch | spacecraft", "Return | spacecraft"]]

,#,Launch | spacecraft,Return | spacecraft
3,—,22 May 2021 [United States] Unity 21,22 May 2021 [United States] Unity 21
64,376,29 June 2025 [United States] NS 33,29 June 2025 [United States] NS 33
66,378,3 August 2025 [United States] NS 34,3 August 2025 [United States] NS 34
67,379,8 October 2025 [United States] NS 36,8 October 2025 [United States] NS 36
70,382,20 December 2025 [United States] NS 37,20 December 2025 [United States] NS 37
71,383,21 January 2026 [United States] NS 38,21 January 2026 [United States] NS 38


.explode() takes a column where each cell is a list and gives every list item its own row. This matters for my analysis because before, every crew member from a launch was crammed into one row, which made it impossible to analyze people individually. After .explode(), each astronaut has their own row, so I can count people per country. The # column is copied onto every new row, so each astronaut stays linked to the launch they flew on.

Note: after exploding, each astronaut kept the row label of the launch row they came from in missions, so several crew members shared the same label (for example, all four Crew-2 astronauts had label 2). That causes problems for anything that assumes one row per label: astronauts.loc[2] would return four rows instead of one. To fix this, I used reset_index(drop=True), which gives every row a new, unique label in order (0, 1, 2, ...) and throws away the old ones.

Starting from 73 launch rows, with most missions carrying 3 to 6 crew members, I hypothesized the table would grow to roughly 3 times its length. The result was 270 rows, about 3.7 times larger. The real number came out a bit higher than predicted because there were more large crews than I expected, even though some rows held only 1 or 2 people.

In [69]:
# one row per astronaut: each person in crew_list gets their own row
# reset_index gives each row a new unique label (0, 1, 2, ...); drop=True throws away the old labels
astronauts = missions.explode("crew_list").reset_index(drop=True)

print("launches:", len(missions))
print("astronaut rows:", len(astronauts))
astronauts[["#", "crew_list"]].head(10)

launches: 73
astronaut rows: 270


,#,crew_list
0,329,[Russia] Oleg Novitsky
1,329,[Russia] Pyotr Dubrov
2,329,[United States] Mark T. Vande Hei
3,330,[United States] Shane Kimbrough
4,330,[United States] K. Megan McArthur
5,330,[Japan] Akihiko Hoshide
6,330,[France] Thomas Pesquet
7,—,[United States] Frederick W. Sturckow [United ...
8,331,[China] Nie Haisheng
9,331,[China] Liu Boming


Filtering out the sub orbital flights:

In [70]:
astronauts.loc[[0, 7], ["#", "Launch | spacecraft", "Return | spacecraft"]]

,#,Launch | spacecraft,Return | spacecraft
0,329,9 April 2021 | [Russia] Soyuz MS-18,17 October 2021 | [Russia] Soyuz MS-18
7,—,22 May 2021 [United States] Unity 21,22 May 2021 [United States] Unity 21


In [71]:
# the date is always the first 3 words (day month year), with or without a "|" after it
launch_date = astronauts["Launch | spacecraft"].str.split().str[:3].str.join(" ")
return_date = astronauts["Return | spacecraft"].str.split().str[:3].str.join(" ")

print(launch_date[[0, 7]])
print(return_date[[0, 7]])

0    9 April 2021
7     22 May 2021
Name: Launch | spacecraft, dtype: object
0    17 October 2021
7        22 May 2021
Name: Return | spacecraft, dtype: object


In [72]:
# True where the astronaut launched and landed on the same day = suborbital
same_day = launch_date == return_date

print("suborbital astronaut rows:", same_day.sum())
astronauts.loc[same_day, ["#", "Launch | spacecraft", "crew_list"]]

suborbital astronaut rows: 128


,#,Launch | spacecraft,crew_list
7,—,22 May 2021 [United States] Unity 21,[United States] Frederick W. Sturckow [United ...
11,—,11 July 2021 [United States] Unity 22,[United Kingdom] David Mackay
12,—,11 July 2021 [United States] Unity 22,[United States] Michael Masucci
13,—,11 July 2021 [United States] Unity 22,[United States] Beth Moses
14,—,11 July 2021 [United States] Unity 22,[United Kingdom] Richard Branson
...,...,...,...
241,376,29 June 2025 [United States] NS 33,[United States] Allie Kuehner[United States] C...
246,378,3 August 2025 [United States] NS 34,[United States] James (J.D.) Russell[Puerto Ri...
247,379,8 October 2025 [United States] NS 36,[United States] Clint Kelly III[United States]...
254,382,20 December 2025 [United States] NS 37,[Germany] Michaela Benthaus[United States] Joe...


In [73]:
# each launch appears once per astronaut, so count DISTINCT launches
sub_launches = astronauts.loc[same_day, "Launch | spacecraft"]
print("suborbital launches:", sub_launches.nunique())
sub_launches.unique()

suborbital launches: 27


array(['22 May 2021 [United States] Unity 21',
       '11 July 2021 [United States] Unity 22',
       '20 July 2021 [United States] NS 16',
       '13 October 2021 [United States] NS 18',
       '11 December 2021 [United States] NS 19',
       '31 March 2022 [United States] NS 20',
       '4 June 2022 [United States] NS 21',
       '4 August 2022 [United States] NS 22',
       '25 May 2023 [United States] Unity 25',
       '29 June 2023 [United States] Galactic 01',
       '10 August 2023 [United States] Galactic 02',
       '8 September 2023 [United States] Galactic 03',
       '6 October 2023 [United States] Galactic 04',
       '2 November 2023 [United States] Galactic 05',
       '26 January 2024 [United States] Galactic 06',
       '19 May 2024 [United States] NS 25',
       '8 June 2024 [United States] Galactic 07',
       '29 August 2024 [United States] NS 26',
       '22 November 2024 [United States] NS 28',
       '25 February 2025 [United States] NS 30',
       '14 April 2025

All suborbital flights launched and returned on the same day, so I used that as my filter. Each launch and return cell starts with the date in "day month year" format, but suborbital rows were missing the "|" after the date, so cutting at the "|" would have broken. Instead, I split each cell on spaces with .str.split() and kept the first three words with [:3], which worked for both formats. Then I compared launch and return dates to make a boolean mask (same_day) that is True for suborbital rows.

The mask flagged 128 astronaut rows from 27 suborbital launches. To make sure it didn't catch any orbital flights, I checked every flagged spacecraft: all were Virgin Galactic (Unity, Galactic) or Blue Origin (New Shepard), the only two vehicles flying people suborbitally.

In [74]:
# keep only orbital flights (NOT same-day), and give rows fresh labels 0, 1, 2...
orbital = astronauts.loc[~same_day].reset_index(drop=True)

print("before:", len(astronauts))
print("after:", len(orbital))

before: 270
after: 142


I removed the suborbital rows with ~same_day, which flips the mask to keep only orbital astronauts, dropping the table from 270 to 142 rows. I filtered before splitting country from name because every row with separator problems was a suborbital flight. Removing them first meant the split could use one rule for all remaining rows.

In [75]:
# split at the LAST "] ": piece 0 = country tag(s), piece 1 = name
orbital[["country", "name"]] = orbital["crew_list"].str.rsplit("] ", n=1, expand=True)

# remove the leftover "[" from the start of each country
orbital["country"] = orbital["country"].str.strip("[")

# collapse double spaces in names (Chun  Wang -> Chun Wang): split on spaces, glue back with one
orbital["name"] = orbital["name"].str.split().str.join(" ")

orbital[["#", "country", "name"]].head(10)

,#,country,name
0,329,Russia,Oleg Novitsky
1,329,Russia,Pyotr Dubrov
2,329,United States,Mark T. Vande Hei
3,330,United States,Shane Kimbrough
4,330,United States,K. Megan McArthur
5,330,Japan,Akihiko Hoshide
6,330,France,Thomas Pesquet
7,331,China,Nie Haisheng
8,331,China,Liu Boming
9,331,China,Tang Hongbo


In [76]:
orbital["country"].value_counts()

,count
country,
United States,48
China,33
Russia,30
Japan,7
France,2
Germany,2
United States] / [Spain,2
Canada,2
Italy,2


To separate country from name, I used .str.rsplit("] ", n=1, expand=True), which cuts only at the last "] ". That always gives exactly two pieces (country, name), even for astronauts with two nationality tags. I then removed the leftover "[" from each country and collapsed double spaces in names.

Using value_counts() on the country column, I found four dual-nationality combinations covering 5 astronaut rows. Since there are only five, I'll look each one up and use the nationality listed for that person in the official mission crew announcement; if they flew under an agency representing a specific country, I'll use that country.

In [77]:
# the 5 dual-nationality astronauts to look up
dual = orbital.loc[orbital["country"].str.contains("/"), ["#", "country", "name", "Launch | spacecraft"]]
dual

,#,country,name,Launch | spacecraft
30,342,United States] / [Spain,Michael López-Alegría,"8 April 2022, | [United States] Axiom Mission ..."
72,356,United States] / [Spain,Michael López-Alegría,"18 January 2024, | [United States] Axiom Missi..."
75,356,Sweden] / [Norway,Marcus Wandt,"18 January 2024, | [United States] Axiom Missi..."
104,370,Malta] / [Saint Kitts and Nevis,Chun Wang,"1 April 2025 | [United States] Fram2, Resilience"
105,370,Norway] / [United Kingdom,Jannicke Mikkelsen,"1 April 2025 | [United States] Fram2, Resilience"


### Dual-nationality decision

Five astronaut rows have two nationality tags. To decide how to count them, I used this rule: each astronaut counts for the country (or countries) they represented on that specific mission, based on official mission sources from the agency or company running the flight. If they flew under an agency representing a specific country, I use that country. If the official sources list both countries, I count both.

Findings so far:
- Marcus Wandt (Ax-3, 2024): listed as Sweden and the European Space Agency → counts as Sweden
- Michael López-Alegría (Ax-1, 2022 and Ax-3, 2024): listed as representing both the United States and Spain on both missions → counts for both countries on both flights
- Chun Wang and Jannicke Mikkelsen (Fram2, 2025): still to look up (Lesson 15)

Limitation: assigning countries to astronauts simplifies nationality, which isn't always a single fact about a person. This analysis measures which countries were represented on each flight, not anyone's personal identity. Counting both countries for dual-represented